# Cap 6: Experimentos Aleatorizados

**Data:** `data.dta` — experimento de aula, 4 semestres, N=70
- Grupo B = definiciones de palabras (tratamiento)
- Grupo A = texto crítico de economía (control)
- `semestre` = estrato (1–4)

**Estructura:**
- PASO 3: Balance de covariables
- PASO 4: Cuatro escenarios de estimación del ATE
- PASO 5: Efectos heterogéneos
- PASO 6: Truco de centrar (Wooldridge)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats as sps
import matplotlib.pyplot as plt

# Cargar datos
df = pd.read_stata('data.dta')
df['y'] = pd.to_numeric(df['resultado'], errors='coerce')
df['D'] = (df['grupo'] == 'B').astype(int)
df['mujer'] = (df['genero'] == 'Mujer').astype(int)
df['pregrado'] = (df['programa'] == 'Pregrado').astype(int)
df['maestria'] = (df['programa'] == 'Maestría').astype(int)
df['semestre'] = df['semestre'].astype(int)

X = ['edad', 'mujer', 'libros', 'pregrado', 'maestria']

print(f'N = {len(df)}')
print(pd.crosstab(df['semestre'], df['D'], margins=True))
print(df.groupby('D')['y'].mean())

## PASO 3: Balance de covariables

In [ ]:
def balance_table(df, vars_list, dcol='D'):
    rows = []
    for v in vars_list:
        g1 = df.loc[df[dcol] == 1, v].dropna().values
        g0 = df.loc[df[dcol] == 0, v].dropna().values
        tstat, pval = sps.ttest_ind(g1, g0, equal_var=False)
        diff = g1.mean() - g0.mean()
        se = diff / tstat if abs(tstat) > 1e-12 else np.nan
        rows.append({
            'Variable': v,
            'Media_T': round(g1.mean(), 3),
            'Media_C': round(g0.mean(), 3),
            'Diff': round(diff, 3),
            't': round(tstat, 3),
            'p-valor': round(pval, 3)
        })
    return pd.DataFrame(rows)

bal = balance_table(df, X)
print('=== Balance univariado ===')
display(bal)
bal.to_excel('Table_Balance_raw.xlsx', index=False)

# Prueba conjunta: D ~ X (LPM con F-test)
f_lpm = 'D ~ ' + ' + '.join(X)
lpm = smf.ols(f_lpm, data=df).fit()
ftest = lpm.f_test(' = '.join([f'{v} = 0' for v in X]))
print(f'\nF-test conjunto: F = {lpm.fvalue:.3f}, p = {lpm.f_pvalue:.3f}')

## PASO 4: Cuatro escenarios de estimación del ATE

In [ ]:
# Crear dummies de semestre
sem_dummies = pd.get_dummies(df['semestre'], prefix='sem', drop_first=True, dtype=int)
df2 = pd.concat([df, sem_dummies], axis=1)
sem_cols = list(sem_dummies.columns)

# Escenario 1: Simple
m1 = smf.ols('y ~ D', data=df2).fit(cov_type='HC1')

# Escenario 2: + Controles
f2 = 'y ~ D + ' + ' + '.join(X)
m2 = smf.ols(f2, data=df2).fit(cov_type='HC1')

# Escenario 3: + Estratos (semestre)
f3 = 'y ~ D + ' + ' + '.join(sem_cols)
m3 = smf.ols(f3, data=df2).fit(cov_type='HC1')

# Escenario 4: Completo (estratos + controles)
f4 = 'y ~ D + ' + ' + '.join(sem_cols) + ' + ' + ' + '.join(X)
m4 = smf.ols(f4, data=df2).fit(cov_type='HC1')

# Tabla comparativa
results = pd.DataFrame({
    'Escenario': ['1: Simple', '2: +Controles', '3: +Estratos', '4: Completo'],
    'tau_hat': [m.params['D'] for m in [m1, m2, m3, m4]],
    'SE(D)':   [m.bse['D'] for m in [m1, m2, m3, m4]],
    'p-valor': [m.pvalues['D'] for m in [m1, m2, m3, m4]],
    'R2':      [m.rsquared for m in [m1, m2, m3, m4]],
    'N':       [int(m.nobs) for m in [m1, m2, m3, m4]]
}).round(4)

print('=== Cuatro escenarios de estimación del ATE ===')
display(results)
results.to_excel('Tabla_4escenarios.xlsx', index=False)

## PASO 5: Efectos heterogéneos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- 5.1 Por mujer (binaria) ---
m_mujer = smf.ols('y ~ D * mujer', data=df).fit(cov_type='HC1')
grid = pd.DataFrame({'D': [0,1,0,1], 'mujer': [0,0,1,1]})
grid['yhat'] = m_mujer.predict(grid)

ax = axes[0]
for i, (d, lbl) in enumerate([(0, 'Control'), (1, 'Tratado')]):
    sub = grid[grid['D'] == d]
    ax.bar(sub['mujer'] + i*0.35 - 0.175, sub['yhat'], width=0.35, label=lbl, alpha=0.8)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Hombre', 'Mujer'])
ax.set_ylabel('E[Y]')
ax.set_title('Efecto por género')
ax.legend()

# --- 5.2 Por libros (continua) ---
m_lib = smf.ols('y ~ D * libros', data=df).fit(cov_type='HC1')
grid_lib = pd.DataFrame({'libros': np.arange(0, 9)})

ax = axes[1]
for d, lbl, c in [(0, 'Control', 'C0'), (1, 'Tratado', 'C1')]:
    g = grid_lib.copy()
    g['D'] = d
    pred = m_lib.get_prediction(g)
    g['yhat'] = pred.predicted_mean
    g['ci_lo'] = pred.conf_int()[:, 0]
    g['ci_hi'] = pred.conf_int()[:, 1]
    ax.plot(g['libros'], g['yhat'], label=lbl)
    ax.fill_between(g['libros'], g['ci_lo'], g['ci_hi'], alpha=0.15)
ax.set_xlabel('Libros leídos')
ax.set_ylabel('E[Y]')
ax.set_title('Efecto según libros')
ax.legend()

# --- 5.3 Por cuartiles de edad ---
df['q_edad'] = pd.qcut(df['edad'].rank(method='first'), 4, labels=['Q1','Q2','Q3','Q4'])
m_q = smf.ols('y ~ D * C(q_edad)', data=df).fit(cov_type='HC1')
grid_q = pd.DataFrame([(d, q) for d in [0,1] for q in ['Q1','Q2','Q3','Q4']],
                       columns=['D', 'q_edad'])
grid_q['q_edad'] = pd.Categorical(grid_q['q_edad'], categories=['Q1','Q2','Q3','Q4'])
grid_q['yhat'] = m_q.predict(grid_q)

ax = axes[2]
qs = ['Q1','Q2','Q3','Q4']
x_pos = np.arange(len(qs))
for i, (d, lbl) in enumerate([(0, 'Control'), (1, 'Tratado')]):
    sub = grid_q[grid_q['D'] == d].set_index('q_edad').loc[qs]
    ax.bar(x_pos + i*0.35 - 0.175, sub['yhat'].values, width=0.35, label=lbl, alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(qs)
ax.set_xlabel('Cuartil de edad')
ax.set_ylabel('E[Y]')
ax.set_title('Heterogeneidad por edad')
ax.legend()

plt.tight_layout()
plt.savefig('het_efectos.png', dpi=300)
plt.show()

## PASO 6: Truco de centrar (Wooldridge)

In [ ]:
# Sin centrar: coef(D) = efecto para libros=0
m_nc = smf.ols('y ~ D * libros', data=df).fit(cov_type='HC1')

# Con centrado: coef(D) = ATE promedio
df['libros_c'] = df['libros'] - df['libros'].mean()
m_c = smf.ols('y ~ D * libros_c', data=df).fit(cov_type='HC1')

comp = pd.DataFrame({
    'Modelo': ['Sin centrar', 'Con centrado'],
    'coef(D)': [m_nc.params['D'], m_c.params['D']],
    'SE(D)': [m_nc.bse['D'], m_c.bse['D']],
    'Interpretación': [
        'Efecto cuando libros = 0',
        'ATE promedio'
    ]
}).round(3)

print('=== Truco de centrar ===')
display(comp)

print(f'\nSin centrar: coef(D) = {m_nc.params["D"]:.3f} → efecto para quien lee 0 libros')
print(f'Con centrado: coef(D) = {m_c.params["D"]:.3f} → ATE promedio')